# T27 -- Phase 3: fixed hourly analysis-timestamp grid (T0..T72)

**Scope.** Pure timestamp-grid generation, `iste.analysis_grid.generate_analysis_grid`.
No particle/KDE/scientific logic here, and no Stage III module is touched -- this task
only produces the fixed set of 73 historical query timestamps (T0 = observed slick,
T72 = 72 hours backward, 1-hour spacing) that T28+ will evaluate diagnostics/KDE/scoring
at. Later timestamps are queried against already-computed Stage III output via the
existing T14 `iste.particle_query.query_particle_slice` (unmodified) -- this task does
not rerun or reinterpolate particle transport itself.

In [1]:
import os, sys
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
from iste.analysis_grid import generate_analysis_grid

## 1. Default 3-day / 1-hour grid (locked Phase 3 configuration)

In [2]:
t_obs = np.datetime64("2019-10-14T23:00:00", "ns")  # RUN_T_OBS from the real-data notebook
grid = generate_analysis_grid(t_obs, window_hours=72.0, interval_hours=1.0)

print(f"n analysis timestamps: {len(grid)}  (expected 73)")
print(f"T0  (observed slick): {grid[0]}")
print(f"T1  (1h backward):    {grid[1]}")
print(f"T72 (72h backward):   {grid[-1]}")
assert len(grid) == 73
assert grid[0] == t_obs
assert grid[-1] == t_obs - np.timedelta64(72, "h")
print("assertions passed: exactly 73 timestamps, T0 == t_obs, T72 == t_obs - 72h")

n analysis timestamps: 73  (expected 73)
T0  (observed slick): 2019-10-14T23:00:00.000000000
T1  (1h backward):    2019-10-14T22:00:00.000000000
T72 (72h backward):   2019-10-11T23:00:00.000000000
assertions passed: exactly 73 timestamps, T0 == t_obs, T72 == t_obs - 72h


In [3]:
diffs_hours = np.diff(grid.astype("int64")) / (3600 * 1_000_000_000)
print(f"spacing (hours) -- min={diffs_hours.min()}, max={diffs_hours.max()} (expect exactly -1.0 everywhere)")
assert np.all(diffs_hours == -1.0)
print(chr(10) + "full grid (all 73 timestamps):")
for k, t in enumerate(grid):
    print(f"  T{k:<3d} = t_obs - {k:>2d}h = {t}")

spacing (hours) -- min=-1.0, max=-1.0 (expect exactly -1.0 everywhere)

full grid (all 73 timestamps):
  T0   = t_obs -  0h = 2019-10-14T23:00:00.000000000
  T1   = t_obs -  1h = 2019-10-14T22:00:00.000000000
  T2   = t_obs -  2h = 2019-10-14T21:00:00.000000000
  T3   = t_obs -  3h = 2019-10-14T20:00:00.000000000
  T4   = t_obs -  4h = 2019-10-14T19:00:00.000000000
  T5   = t_obs -  5h = 2019-10-14T18:00:00.000000000
  T6   = t_obs -  6h = 2019-10-14T17:00:00.000000000
  T7   = t_obs -  7h = 2019-10-14T16:00:00.000000000
  T8   = t_obs -  8h = 2019-10-14T15:00:00.000000000
  T9   = t_obs -  9h = 2019-10-14T14:00:00.000000000
  T10  = t_obs - 10h = 2019-10-14T13:00:00.000000000
  T11  = t_obs - 11h = 2019-10-14T12:00:00.000000000
  T12  = t_obs - 12h = 2019-10-14T11:00:00.000000000
  T13  = t_obs - 13h = 2019-10-14T10:00:00.000000000
  T14  = t_obs - 14h = 2019-10-14T09:00:00.000000000
  T15  = t_obs - 15h = 2019-10-14T08:00:00.000000000
  T16  = t_obs - 16h = 2019-10-14T07:00:00.000000

## 2. Cross-check: does the grid match the real Stage III checkpoints already computed?

The real-data notebook (`notebooks/main_actual_data_pipeline.ipynb`) already ran Stage III
with `dt=3600s` (1 hour) over 72 steps, so its checkpoint schedule should coincide exactly
with this task's generated analysis grid -- confirming no second/inconsistent timestamp
scheme is being introduced.

In [4]:
history_path = os.path.join(REPO_ROOT, "results", "actual_data", "stage3_particle_history.npz")
if os.path.exists(history_path):
    cached = np.load(history_path, allow_pickle=True)
    real_checkpoints = np.unique(cached["timestamp"])
    print(f"real Stage III checkpoints on disk: {len(real_checkpoints)}")
    matches = bool(np.array_equal(np.sort(grid), real_checkpoints))
    print(f"T27 grid matches real Stage III checkpoints exactly: {matches}")
    assert matches
else:
    print(f"no cached real Stage III result found at {history_path} -- skipping cross-check")

real Stage III checkpoints on disk: 73
T27 grid matches real Stage III checkpoints exactly: True


## 3. Non-default configuration example (demonstrates the parameters are not hard-coded)

In [5]:
small_grid = generate_analysis_grid(t_obs, window_hours=6.0, interval_hours=2.0)
print(f"window_hours=6, interval_hours=2 -> {len(small_grid)} timestamps (expect 4):")
for k, t in enumerate(small_grid):
    print(f"  {t}")
assert len(small_grid) == 4

window_hours=6, interval_hours=2 -> 4 timestamps (expect 4):
  2019-10-14T23:00:00.000000000
  2019-10-14T21:00:00.000000000
  2019-10-14T19:00:00.000000000
  2019-10-14T17:00:00.000000000


## 4. Validation errors on malformed configuration

In [6]:
try:
    generate_analysis_grid(t_obs, window_hours=72.0, interval_hours=5.0)
    print("FAIL: expected ValueError")
except ValueError as e:
    print(f"correctly rejected non-divisible window: {e}")

try:
    generate_analysis_grid(t_obs, window_hours=-1.0, interval_hours=1.0)
    print("FAIL: expected ValueError")
except ValueError as e:
    print(f"correctly rejected non-positive window_hours: {e}")

correctly rejected non-divisible window: window_hours (72.0) must be an exact multiple of interval_hours (5.0); got 14.4 steps, which is not an integer
correctly rejected non-positive window_hours: window_hours must be > 0, got -1.0


## Known limitations (T27 scope only)

- This task produces the timestamp grid only. It does not compute any diagnostic, KDE,
  normalization, persistence, convergence, dispersion, screening, AIS, or composite-score
  logic -- those are T28-T39.
- No Stage III module (`iste/step.py`, `iste/trajectory.py`, `iste/ensemble_run.py`) was
  read or modified by this task.
- The physical Stage III integration timestep is unrelated to and unconstrained by this
  grid; the real-data run happens to also use 1-hour physical checkpoints, which is why
  the cross-check above matches exactly, but this module would work identically if the
  physical checkpoints were finer (T14's existing interpolation handles that case, not
  this module).